In [7]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import glob
from xgrads import open_CtlDataset
from pathlib import Path
import netCDF4

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib.colors as mcolors

import ipywidgets as widgets
from IPython.display import display, clear_output

ncl_cmap = LinearSegmentedColormap.from_list(
    "BlueWhiteOrangeRed",
    ["#2166ac", "#67a9cf", "#ffffff", "#fdae61", "#b2182b"],
    N=256
)


plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_colwidth", 160)

print("Python OK")

Python OK


In [8]:
from pathlib import Path

BASE_DIR = Path.cwd()

NEW_DIR = BASE_DIR / ".." / ".." / ".." /"SPEEDY_access" / "output" / "exp_102"
OLD_DIR = BASE_DIR / ".." / ".." / ".." /"SPEEDY_access" / "output" / "exp_101"

print("NEW_DIR:", NEW_DIR, NEW_DIR.exists())
print("OLD_DIR:", OLD_DIR, OLD_DIR.exists())

# SPEEDY -> ACCESS-OM2 variable names
SPEEDY_VARIABLES = {
    "SLRD": "rlds",
    "SSRD": "rsds",
}

# Output directory
FORCING_OUT_DIR = (BASE_DIR / ".." / ".." / ".." / "SPEEDY_access" / "access_forcing").resolve()
FORCING_OUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_ONE_FILE_PER_YEAR = True

# Keep SPEEDY grid untouched until JRA-55 metadata/grid are inspected
SHIFT_LONGITUDE_TO_MINUS180_180 = False
SORT_LATITUDE_NORTH_TO_SOUTH = False

print("Output directory:", FORCING_OUT_DIR)
print("Variables:", SPEEDY_VARIABLES)

NEW_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_102 True
OLD_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_101 True
Output directory: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing
Variables: {'SLRD': 'rlds', 'SSRD': 'rsds'}


In [9]:
for ctl in Path(NEW_DIR).glob("attm102.ctl"):
    print("=" * 80)
    print(ctl.name)

    ds = open_CtlDataset(str(ctl))
#ds

attm102.ctl


In [10]:
# Check SPEEDY variables

for speedy_var, access_var in SPEEDY_VARIABLES.items():

    if speedy_var not in ds:
        raise KeyError(
            f"{speedy_var!r} is absent from the SPEEDY dataset. "
            f"Available variables: {list(ds.data_vars)}"
        )

    var = ds[speedy_var]

    print(f"\n{'='*60}")
    print(f"{speedy_var} -> {access_var}")
    print(f"{'='*60}")

    print(var)
    print("Dimensions:", var.dims)
    print("Shape:", var.shape)
    print("Dtype:", var.dtype)
    print("Attributes:", var.attrs)

    # print(
    #     f"{speedy_var} range:",
    #     float(var.min().compute()),
    #     "to",
    #     float(var.max().compute()),
    # )

    # print(
    #     f"{speedy_var} global mean:",
    #     float(var.mean(skipna=True).compute()),
    # )

# Check temporal resolution only once
dt_hours = np.diff(ds.time.values) / np.timedelta64(1, "h")
print("\nUnique output intervals [hours]:", np.unique(dt_hours))


SLRD -> rlds
<xarray.DataArray 'SLRD' (time: 8760, lat: 48, lon: 96)> Size: 161MB
dask.array<reshape, shape=(8760, 48, 96), dtype=>f4, chunksize=(1, 48, 96), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Attributes:
    comment:  surface LW rad.
    storage:  99
Dimensions: ('time', 'lat', 'lon')
Shape: (8760, 48, 96)
Dtype: >f4
Attributes: {'comment': 'surface LW rad.', 'storage': '99'}

SSRD -> rsds
<xarray.DataArray 'SSRD' (time: 8760, lat: 48, lon: 96)> Size: 161MB
dask.array<reshape, shape=(8760, 48, 96), dtype=>f4, chunksize=(1, 48, 96), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75

In [11]:
# Build ACCESS-OM2 forcing variables

forcing_vars = {}

for speedy_var, access_var in SPEEDY_VARIABLES.items():
    var = ds[speedy_var].rename(access_var).astype("float32")

    if SHIFT_LONGITUDE_TO_MINUS180_180:
        var = var.assign_coords(lon=((var.lon + 180.0) % 360.0) - 180.0).sortby("lon")
    if SORT_LATITUDE_NORTH_TO_SOUTH:
        var = var.sortby("lat", ascending=False)

    if access_var == "rlds":
        var.attrs = {"standard_name": "surface_downwelling_longwave_flux_in_air", "long_name": "Surface Downwelling Longwave Radiation", "units": "W m-2", "cell_methods": "area: time: mean", "source_variable": "SLRD", "source_model": "SPEEDY", "mapping_note": "SPEEDY SLRD -> ACCESS-OM2 rlds"}
    elif access_var == "rsds":
        var.attrs = {"standard_name": "surface_downwelling_shortwave_flux_in_air", "long_name": "Surface Downwelling Shortwave Radiation", "units": "W m-2", "cell_methods": "area: time: mean", "source_variable": "SSRD", "source_model": "SPEEDY", "mapping_note": "SPEEDY SSRD -> ACCESS-OM2 rsds"}

    forcing_vars[access_var] = var

forcing_ds = xr.Dataset(forcing_vars)

# SPEEDY raw times mark the start of each 3-hour averaging interval.
# Follow JRA55-do convention: time = interval midpoint, with explicit bounds.
raw_time = forcing_ds.time.copy()
forcing_ds = forcing_ds.assign_coords(time=raw_time + np.timedelta64(90, "m"))
forcing_ds["time_bnds"] = xr.DataArray(
    np.stack([raw_time.values, (raw_time + np.timedelta64(3, "h")).values], axis=1),
    dims=("time", "bnds"), coords={"time": forcing_ds.time, "bnds": [0, 1]}
)

forcing_ds["lat"].attrs.update({"standard_name": "latitude", "long_name": "Latitude", "units": "degrees_north", "axis": "Y"})
forcing_ds["lon"].attrs.update({"standard_name": "longitude", "long_name": "Longitude", "units": "degrees_east", "axis": "X"})
forcing_ds["time"].attrs.update({"standard_name": "time", "long_name": "time", "axis": "T", "bounds": "time_bnds"})

forcing_ds.attrs = {
    "Conventions": "CF-1.7",
    "title": "SPEEDY forcing for ACCESS-OM2",
    "source": "SPEEDY model output",
    "frequency": "3hr",
    "history": "Created from SPEEDY SSRD and SLRD",
    "comment": "3-hour mean radiation forcing. SPEEDY SSRD -> rsds, SLRD -> rlds; time follows JRA55-do midpoint convention.",
}

forcing_ds

<xarray.Dataset> Size: 323MB
Dimensions:    (lat: 48, lon: 96, time: 8760, bnds: 2)
Coordinates:
  * lat        (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon        (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
  * time       (time) datetime64[ns] 70kB 1989-01-01T01:30:00 ... 1991-12-31T...
  * bnds       (bnds) int64 16B 0 1
Data variables:
    rlds       (time, lat, lon) float32 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    rsds       (time, lat, lon) float32 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    time_bnds  (time, bnds) datetime64[ns] 140kB 1989-01-01 ... 1992-01-01
Attributes:
    Conventions:  CF-1.7
    title:        SPEEDY forcing for ACCESS-OM2
    source:       SPEEDY model output
    frequency:    3hr
    history:      Created from SPEEDY SSRD and SLRD
    comment:      3-hour mean radiation forcing. SPEEDY SSRD -> rsds, SLRD ->...

In [ ]:
required_dims = ("time", "lat", "lon")
expected_units = {"rlds": "W m-2", "rsds": "W m-2"}

for var_name in ["rlds", "rsds"]:
    var = forcing_ds[var_name]

    if var.dims != required_dims:
        raise ValueError(f"Expected {var_name} dimensions {required_dims}, got {var.dims}")
    if var.attrs.get("units") != expected_units[var_name]:
        raise ValueError(f"{var_name} must have units {expected_units[var_name]!r}, got {var.attrs.get('units')!r}")
    if var.attrs.get("cell_methods") != "area: time: mean":
        raise ValueError(f"{var_name} must have cell_methods='area: time: mean'")
    if not np.issubdtype(var.dtype, np.floating):
        raise TypeError(f"{var_name} must be floating point, got {var.dtype}")

    invalid_count = int((~np.isfinite(var)).sum().compute())
    if invalid_count:
        raise ValueError(f"Found {invalid_count} invalid values in {var_name}")

    var_min, var_max = float(var.min().compute()), float(var.max().compute())

    if var_min < 0.0:
        print(f"WARNING: negative {var_name}: {var_min:.3f} W m-2")
    if var_name == "rlds" and var_max > 700.0:
        print(f"WARNING: unusually high rlds: {var_max:.3f} W m-2")
    if var_name == "rsds" and var_max > 1500.0:
        print(f"WARNING: unusually high rsds: {var_max:.3f} W m-2")

    print(f"{var_name}: {var_min:.3f} to {var_max:.3f} W m-2")

# Time checks
dt_hours = np.diff(forcing_ds.time.values) / np.timedelta64(1, "h")
if not np.all(dt_hours == 3):
    raise ValueError(f"Expected 3-hourly output, got {np.unique(dt_hours)} hours")

if forcing_ds["time"].attrs.get("bounds") != "time_bnds":
    raise ValueError("time must reference time_bnds")

b0 = forcing_ds.time_bnds.isel(bnds=0).values
b1 = forcing_ds.time_bnds.isel(bnds=1).values
width_hours = (b1 - b0) / np.timedelta64(1, "h")
midpoint = b0 + (b1 - b0) / 2

if not np.all(width_hours == 3):
    raise ValueError(f"Expected 3-hour time bounds, got {np.unique(width_hours)} hours")
if not np.all(forcing_ds.time.values == midpoint):
    raise ValueError("time coordinate is not the midpoint of time_bnds")

print("\nValidation passed")
print("time:", forcing_ds.time.values[0], "to", forcing_ds.time.values[-1])
print("first bounds:", b0[0], "to", b1[0])
print("grid:", forcing_ds.sizes["lat"], "x", forcing_ds.sizes["lon"])
print("time interval:", np.unique(dt_hours), "hours")

In [12]:
field_encoding = {"dtype": "float32", "zlib": True, "complevel": 4, "shuffle": True, "_FillValue": np.float32(1.0e20), "chunksizes": (1, forcing_ds.sizes["lat"], forcing_ds.sizes["lon"])}

encoding = {
    **{v: field_encoding.copy() for v in ["rlds", "rsds"]},
    "time": {"dtype": "float64", "units": "days since 1900-01-01 00:00:00", "calendar": "gregorian", "_FillValue": None},
    "time_bnds": {"dtype": "float64", "units": "days since 1900-01-01 00:00:00", "calendar": "gregorian", "_FillValue": None},
    "lat": {"dtype": "float64", "_FillValue": None},
    "lon": {"dtype": "float64", "_FillValue": None},
}

In [13]:
written_files = []

if WRITE_ONE_FILE_PER_YEAR:
    years = np.unique(forcing_ds.time.dt.year.values)

    for year in years:
        yearly = forcing_ds.sel(time=str(int(year)))

        for var_name in ["rlds", "rsds"]:
            var_ds = yearly[[var_name, "time_bnds"]]
            output_file = FORCING_OUT_DIR / f"{var_name}_SPEEDY_{int(year)}.nc"

            var_encoding = {
                var_name: encoding[var_name],
                "time": encoding["time"],
                "time_bnds": encoding["time_bnds"],
                "lat": encoding["lat"],
                "lon": encoding["lon"],
            }

            var_ds.to_netcdf(output_file, mode="w", format="NETCDF4", engine="netcdf4",
                             unlimited_dims=["time"], encoding=var_encoding)

            written_files.append(output_file)
            print(f"Wrote {output_file.name}: {var_ds.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

else:
    for var_name in ["rlds", "rsds"]:
        var_ds = forcing_ds[[var_name, "time_bnds"]]
        output_file = FORCING_OUT_DIR / f"{var_name}_SPEEDY_all_years.nc"

        var_encoding = {
            var_name: encoding[var_name],
            "time": encoding["time"],
            "time_bnds": encoding["time_bnds"],
            "lat": encoding["lat"],
            "lon": encoding["lon"],
        }

        var_ds.to_netcdf(output_file, mode="w", format="NETCDF4", engine="netcdf4",
                         unlimited_dims=["time"], encoding=var_encoding)

        written_files.append(output_file)
        print(f"Wrote {output_file.name}: {var_ds.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

written_files

Wrote rlds_SPEEDY_1989.nc: 2920 records, 38.80 MB
Wrote rsds_SPEEDY_1989.nc: 2920 records, 36.45 MB
Wrote rlds_SPEEDY_1990.nc: 2920 records, 38.86 MB
Wrote rsds_SPEEDY_1990.nc: 2920 records, 36.45 MB
Wrote rlds_SPEEDY_1991.nc: 2920 records, 38.92 MB
Wrote rsds_SPEEDY_1991.nc: 2920 records, 36.44 MB


[PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/rlds_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/rsds_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/rlds_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/rsds_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/rlds_SPEEDY_1991.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/rsds_SPEEDY_1991.nc')]

In [ ]:
if not written_files:
    raise RuntimeError("No NetCDF files were written")

for check_file in written_files:
    var_name = check_file.name.split("_")[0]

    with xr.open_dataset(check_file, decode_times=True) as check:
        print(f"\n{'='*60}\n{check_file.name}\n{'='*60}")
        print(check)

        print("\nVariable attributes:")
        print(check[var_name].attrs)

        print("\nEncoding:")
        print(check[var_name].encoding)

        print("\nTime:")
        print(check.time.values[0], "to", check.time.values[-1])
        print("First bounds:", check.time_bnds.values[0])

        dt_hours = np.diff(check.time.values) / np.timedelta64(1, "h")
        bounds_hours = (check.time_bnds[:,1] - check.time_bnds[:,0]).values / np.timedelta64(1, "h")
        midpoint = check.time_bnds[:,0].values + (check.time_bnds[:,1].values - check.time_bnds[:,0].values) / 2

        if not np.all(dt_hours == 3):
            raise ValueError(f"Unexpected time interval: {np.unique(dt_hours)} h")
        if not np.all(bounds_hours == 3):
            raise ValueError(f"Unexpected time bounds: {np.unique(bounds_hours)} h")
        if not np.all(check.time.values == midpoint):
            raise ValueError("time is not midpoint of time_bnds")

        var_min, var_max = float(check[var_name].min()), float(check[var_name].max())
        units = check[var_name].attrs.get("units", "")
        print(f"\nRange [{units}]: {var_min:.6f} to {var_max:.6f}")

        year = int(check.time.dt.year.values[0])
        source_first = forcing_ds[var_name].sel(time=str(year)).isel(time=0).compute()
        output_first = check[var_name].isel(time=0).load()
        max_abs_difference = float(np.abs(source_first - output_first).max())

        print("Maximum absolute difference after NetCDF round trip:", max_abs_difference)

print("\nAll output files verified")